# Phase 4 — QLoRA fine-tune of Qwen2.5-3B-Instruct

Distills Claude's citation-grounded answering behaviour (Phase 2's synthetic QA set,
prepared for training by `training/prepare_dataset.py`) into a small open model via
QLoRA, on a free Colab T4.

**Before running this notebook:** run `training/prepare_dataset.py` locally (or in a
CPU-only Colab cell) to produce `training/data/{train,val,test}.jsonl`, then upload
those three files here (or push them to a private repo/Drive folder and adjust
`DATA_DIR` below). They are small (~1-2 MB) — no need to re-run retrieval on Colab.

**Sizing notes baked into this design (see `derma_guide_plan.md` Phase 4 for the reasoning):**
- T4 is Turing: **no bf16, no FlashAttention-2** — this notebook uses fp16 compute
  throughout.
- `max_seq_len=2048`, not the originally-planned 1024 — dermatology chunk text tokenizes
  to a median ~600 tokens (heavy Latin/technical vocabulary), so 1024 only ever fit one
  retrieved context chunk. 2048 reliably fits 2-3.
- Required hyperparameter experiment: LoRA rank **r ∈ {8, 16, 32}**, run sequentially,
  each ~40-60 min. **This notebook checkpoints to Google Drive** — free Colab
  disconnects, and losing a 50-minute run is the most likely way this phase slips.
- Training targets include deliberate **abstention examples** (see `prepare_dataset.py`):
  wherever retrieval didn't surface the source chunk within the token budget, the target
  was swapped to an explicit "not enough information" refusal rather than the original
  citation-grounded answer — training on the original answer there would teach the model
  to state facts not present in its shown context.


## 1. Setup

In [ ]:
!pip install -q -U "transformers>=4.44,<5" "accelerate>=0.33" "peft>=0.12" "bitsandbytes>=0.43" datasets

# IMPORTANT: Colab pre-imports an older transformers for its own tooling before this
# cell ever runs. `pip install -U` only updates the on-disk package — the process
# already has the old module cached in sys.modules. If the next cell's version assert
# fails (or training later fails with a mysterious "unexpected keyword argument" on a
# perfectly normal TrainingArguments field like warmup_ratio), that's this issue:
# Runtime > Restart session, then Runtime > Run all again. Re-running cells in place
# without restarting will NOT pick up the new install.


In [ ]:
import json
import os
import time
from pathlib import Path

import torch
import transformers
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

# Fail loudly HERE, not 12 cells later inside the training loop, if Colab's stale
# pre-imported transformers is still the one in memory (see cell above). Any 4.44+
# has every TrainingArguments field this notebook uses.
_tf_version = tuple(int(x) for x in transformers.__version__.split(".")[:2])
assert _tf_version >= (4, 44), (
    f"transformers {transformers.__version__} is loaded, but this notebook needs "
    f">=4.44 (pip installed it, but Colab likely had an older copy already imported). "
    f"Runtime > Restart session, then Runtime > Run all — do not just re-run this cell."
)
print("transformers", transformers.__version__)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/derma_force_qlora')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoints and logs will be saved under:", DRIVE_DIR)


## 2. Load the prepared training data

Upload `train.jsonl`, `val.jsonl` (and optionally `test.jsonl`, not used for training)
via the Colab file browser into `/content/`, or point `DATA_DIR` at wherever you've
placed them (e.g. a Drive folder).

In [ ]:
DATA_DIR = Path('/content/drive/MyDrive/derma_force_qlora')  # adjust if you uploaded elsewhere
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LEN = 2048  # matches training/prepare_dataset.py — must stay in sync

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_records = load_jsonl(DATA_DIR / 'train.jsonl')
val_records = load_jsonl(DATA_DIR / 'val.jsonl')
print(f"train: {len(train_records)}  val: {len(val_records)}")

n_grounded = sum(1 for r in train_records if r['target_type'] == 'grounded')
print(f"train target types: {n_grounded} grounded, {len(train_records)-n_grounded} abstention")


## 3. Tokenizer + loss-masked tokenization

Only the assistant's completion tokens contribute to the loss — the system prompt and
user message (question + retrieved excerpts) are masked with `-100` so the model isn't
trained to predict its own input.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # fallback only — Qwen2.5-3B-Instruct already ships a distinct pad token ('<|endoftext|>'), this just guards other base models

def tokenize_example(example):
    messages = example['messages']
    prompt_text = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)['input_ids']
    full_ids = tokenizer(full_text, add_special_tokens=False)['input_ids']

    if len(full_ids) > MAX_SEQ_LEN:
        full_ids = full_ids[:MAX_SEQ_LEN]
    prompt_len = min(len(prompt_ids), len(full_ids))

    labels = [-100] * prompt_len + full_ids[prompt_len:]
    return {
        'input_ids': full_ids,
        'attention_mask': [1] * len(full_ids),
        'labels': labels,
    }

def build_dataset(records):
    ds = Dataset.from_list(records)
    ds = ds.map(tokenize_example, remove_columns=ds.column_names)
    return ds

train_ds = build_dataset(train_records)
val_ds = build_dataset(val_records)
print(train_ds)


In [ ]:
def collate(batch):
    max_len = max(len(ex['input_ids']) for ex in batch)
    pad_id = tokenizer.pad_token_id

    input_ids, attention_mask, labels = [], [], []
    for ex in batch:
        n_pad = max_len - len(ex['input_ids'])
        input_ids.append(ex['input_ids'] + [pad_id] * n_pad)
        attention_mask.append(ex['attention_mask'] + [0] * n_pad)
        labels.append(ex['labels'] + [-100] * n_pad)

    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.long),
    }


## 4. Model loading (4-bit NF4) + LoRA config

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 is Turing: no bf16
    bnb_4bit_use_double_quant=True,
)

def load_base_model():
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    return model

QWEN_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

def make_lora_config(r):
    return LoraConfig(
        r=r,
        lora_alpha=2 * r,
        lora_dropout=0.05,
        target_modules=QWEN_TARGET_MODULES,
        bias="none",
        task_type="CAUSAL_LM",
    )


## 5. Loss-curve CSV logging

Trainer's own log history has both train and eval entries interleaved; this callback
writes each to a per-rank CSV as training proceeds, so a checkpoint that gets
interrupted mid-run still leaves a usable partial curve on Drive.

In [ ]:
class CSVLossLogger(TrainerCallback):
    def __init__(self, csv_path):
        self.csv_path = csv_path
        # Don't truncate: if this rank is resuming from a Drive checkpoint after a
        # disconnect, the pre-disconnect rows are still valid loss history and
        # should be kept, not wiped.
        if not Path(csv_path).exists():
            with open(self.csv_path, 'w', encoding='utf-8') as f:
                f.write('step,split,loss\n')

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        with open(self.csv_path, 'a', encoding='utf-8') as f:
            if 'loss' in logs:
                f.write(f"{state.global_step},train,{logs['loss']}\n")
            if 'eval_loss' in logs:
                f.write(f"{state.global_step},val,{logs['eval_loss']}\n")


## 6. Train the LoRA rank sweep (r = 8, 16, 32)

Reloads the base model fresh for each rank rather than swapping adapters in-place —
slower (~1-2 min extra per run) but avoids any risk of adapter state leaking between
runs. Each run's adapter + loss CSV is saved to Drive as it finishes, so a disconnect
after run *k* doesn't lose runs 1..k-1.

**Disconnect-safe:** re-running this cell after `Runtime > Run all` (following a
disconnect) skips any rank whose `adapter/` already exists on Drive, and resumes an
interrupted rank from its latest `checkpoints/checkpoint-N` (Trainer's own
optimizer/scheduler/RNG state included) instead of restarting it from step 0.


In [ ]:
LORA_RANKS = [8, 16, 32]
RESULTS = {}

for r in LORA_RANKS:
    run_dir = DRIVE_DIR / f"r{r}"
    adapter_dir = run_dir / "adapter"
    if adapter_dir.exists() and any(adapter_dir.iterdir()):
        print(f"r={r}: adapter already saved at {adapter_dir} — skipping (already finished before a prior disconnect).")
        continue

    print(f"\n{'='*60}\nTraining LoRA r={r}\n{'='*60}")
    run_dir.mkdir(parents=True, exist_ok=True)
    csv_path = run_dir / "loss.csv"
    checkpoints_dir = run_dir / "checkpoints"

    # Resume this rank from its latest Drive checkpoint, if a previous attempt got
    # partway through before disconnecting. `checkpoint-N` directories carry the
    # optimizer/scheduler/RNG state, not just weights, so this is a true resume,
    # not a restart with a warm init.
    resume_from = None
    if checkpoints_dir.exists():
        ckpts = sorted(checkpoints_dir.glob("checkpoint-*"), key=lambda p: int(p.name.rsplit("-", 1)[-1]))
        if ckpts:
            resume_from = str(ckpts[-1])
            print(f"Resuming r={r} from {resume_from}")

    model = load_base_model()
    model = get_peft_model(model, make_lora_config(r))
    model.print_trainable_parameters()

    args = TrainingArguments(
        output_dir=str(checkpoints_dir),
        per_device_train_batch_size=1,
        # Eval defaults to per_device_eval_batch_size=8 if unset. At max_seq_len=2048
        # and Qwen2.5's ~152k vocab, the float32 upcast in the loss computation
        # (batch x seq_len x vocab x 4 bytes) alone needs ~8-10 GiB for a batch of 8
        # — that OOMs a 14.56 GiB T4 on top of the ~9 GiB the quantized model +
        # optimizer already hold. Match it to the train batch size.
        per_device_eval_batch_size=1,
        # We only ever read eval_loss (via CSVLossLogger) — never eval predictions —
        # so skip retaining logits during evaluation entirely.
        prediction_loss_only=True,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        fp16=True,
        gradient_checkpointing=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collate,
        callbacks=[CSVLossLogger(csv_path)],
    )

    t0 = time.time()
    trainer.train(resume_from_checkpoint=resume_from)
    elapsed = time.time() - t0
    print(f"r={r} done in {elapsed/60:.1f} min")

    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    RESULTS[r] = {"elapsed_min": elapsed / 60, "csv_path": str(csv_path)}

    del model, trainer
    torch.cuda.empty_cache()

print("\nAll ranks done:", RESULTS)


## 7. Loss curves

The plot the Analysis section is built on: val loss should decrease then flatten for
all three ranks, and r=32 (most capacity, most prone to overfitting a ~1000-example
set) is the rank most likely to show the earliest val-loss uptick.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
for r in LORA_RANKS:
    df = pd.read_csv(DRIVE_DIR / f"r{r}" / "loss.csv")
    train_df = df[df.split == 'train']
    val_df = df[df.split == 'val']
    ax.plot(train_df.step, train_df.loss, label=f"r={r} train", alpha=0.5, linestyle='--')
    ax.plot(val_df.step, val_df.loss, label=f"r={r} val", linewidth=2)

ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.set_title("QLoRA fine-tune: train/val loss by LoRA rank")
ax.legend()
fig.tight_layout()
fig.savefig(DRIVE_DIR / "loss_curves.png", dpi=150)
plt.show()
print("Saved ->", DRIVE_DIR / "loss_curves.png")


## Next steps (Phase 5)

Adapters are saved under `DRIVE_DIR/r{8,16,32}/adapter/`. Pick the winning rank from the
loss curves above (lowest val loss without a clear overfitting uptick) and load it in
Phase 5's generation eval as the `LoRAGenerator` arm (`backend/generators.py`), alongside
the base-model-no-RAG, base-model+RAG, and Claude+RAG arms on the same 200-item test
set.